# Student Data Quality Analysis

## Context & Objectives

  A university is preparing for a major student system transformation.
  Data exists across three systems (SRS, Finance, LMS) and reporting
  outputs do not align. This notebook profiles the data landscape,
  identifies and categorises data quality issues, and investigates
  root causes to support prioritisation.

**Systems:** Student Record System · Finance · Learning Management System.

**Analytical framework:** HESA compliance · data completeness ·
  accuracy · consistency

## 1. Setup & Data Loading

In [1]:
import duckdb

DATA    = r"E:/data_analytics/jobs/projects/pact-stage2/data/"
DB_PATH = r"E:/data_analytics/jobs/projects/pact-stage2/data/student.duckdb"

con = duckdb.connect(DB_PATH)

tables = {
    'srs': 'srs.csv',
    'finance': 'finance.csv',
    'lms': 'lms.csv'
}

for name, file in tables.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE {name}
        AS SELECT * FROM read_csv_auto('{DATA + file}')      
    """)
    
    print(f' loaded: {name}')

print('\nDatabase ready.')


 loaded: srs
 loaded: finance
 loaded: lms

Database ready.


 ## 2. Data Understanding — Structure

In [2]:
con.sql("""
    SELECT 
        'srs' AS system, 
        COUNT(*) AS row_count 
    FROM srs     
      
    UNION ALL
      
    SELECT 
        'finance' AS system, 
        COUNT(*) AS row_count 
    FROM finance  
      
    UNION ALL
      
    SELECT 
        'lms' AS system, 
        COUNT(*) AS row_count 
    FROM lms
  """).show()


┌─────────┬───────────┐
│ system  │ row_count │
│ varchar │   int64   │
├─────────┼───────────┤
│ srs     │     15075 │
│ finance │     15450 │
│ lms     │     14100 │
└─────────┴───────────┘



### Insights

- Differing row counts between systems.

In [3]:
con.sql("""
    SELECT
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_name IN ('srs', 'finance', 'lms')
    ORDER BY table_name
""").show(max_rows=28)

┌────────────┬────────────────────┬───────────┐
│ table_name │    column_name     │ data_type │
│  varchar   │      varchar       │  varchar  │
├────────────┼────────────────────┼───────────┤
│ finance    │ student_ref        │ VARCHAR   │
│ finance    │ first_name         │ VARCHAR   │
│ finance    │ last_name          │ VARCHAR   │
│ finance    │ date_of_birth      │ DATE      │
│ finance    │ fee_status         │ VARCHAR   │
│ finance    │ tuition_fee_amount │ BIGINT    │
│ finance    │ bursary_flag       │ VARCHAR   │
│ finance    │ payment_status     │ VARCHAR   │
│ finance    │ enrolment_status   │ VARCHAR   │
│ finance    │ academic_year      │ VARCHAR   │
│ lms        │ last_login_date    │ DATE      │
│ lms        │ username           │ VARCHAR   │
│ lms        │ programme          │ VARCHAR   │
│ lms        │ modules_enrolled   │ BIGINT    │
│ lms        │ lms_id             │ VARCHAR   │
│ lms        │ account_status     │ VARCHAR   │
│ srs        │ student_id         │ VARC

### Inisghts

- Each system uses a different column name for the student identifier: 
    - srs - student_id 
    - lms - lms_id
    - finance - student_ref
    - Therefore, linkage across systems is unknown until the values are compared.
- Shared fields:
    - enrolment_status — SRS and Finance (LMS uses account_status instead — different name for the same concept).  
    - fee_status — SRS and Finance only.
    - first_name, last_name, date_of_birth — SRS and Finance only, not LMS.  
    - programme — SRS and LMS only, named differently (programme_name in SRS, programme in LMS). Not present in Finance.  
- LMS is the thinnest system (6 fields vs 10–12) — least data to cross-reference against.

## 3. Data Understanding — Quality

### 3.1 Null Rates  

In [4]:
# SRS
con.sql("""
    SELECT 
        COUNT(*) - COUNT(student_id)       AS student_id_nulls,
        COUNT(*) - COUNT(first_name)       AS first_name_nulls,
        COUNT(*) - COUNT(last_name)        AS last_name_nulls,
        COUNT(*) - COUNT(date_of_birth)    AS dob_nulls,
        COUNT(*) - COUNT(nationality)      AS nationality_nulls,
        COUNT(*) - COUNT(domicile)         AS domicile_nulls,
        COUNT(*) - COUNT(programme_code)   AS programme_code_nulls,
        COUNT(*) - COUNT(enrolment_status) AS enrolment_status_nulls,
        COUNT(*) - COUNT(fee_status)       AS fee_status_nulls
    FROM srs
""").show()

┌──────────────────┬──────────────────┬─────────────────┬───────────┬───────────────────┬────────────────┬──────────────────────┬────────────────────────┬──────────────────┐
│ student_id_nulls │ first_name_nulls │ last_name_nulls │ dob_nulls │ nationality_nulls │ domicile_nulls │ programme_code_nulls │ enrolment_status_nulls │ fee_status_nulls │
│      int64       │      int64       │      int64      │   int64   │       int64       │     int64      │        int64         │         int64          │      int64       │
├──────────────────┼──────────────────┼─────────────────┼───────────┼───────────────────┼────────────────┼──────────────────────┼────────────────────────┼──────────────────┤
│                0 │                0 │               0 │         0 │              1511 │              0 │                    0 │                      0 │                0 │
└──────────────────┴──────────────────┴─────────────────┴───────────┴───────────────────┴────────────────┴──────────────────────┴─

In [5]:
# Finance
con.sql("""
    SELECT      
        COUNT(*) - COUNT(student_ref)        AS student_ref_nulls,
        COUNT(*) - COUNT(first_name)         AS first_name_nulls,      
        COUNT(*) - COUNT(last_name)          AS last_name_nulls,
        COUNT(*) - COUNT(date_of_birth)      AS dob_nulls,
        COUNT(*) - COUNT(fee_status)         AS fee_status_nulls,
        COUNT(*) - COUNT(tuition_fee_amount) AS tuition_fee_nulls,
        COUNT(*) - COUNT(bursary_flag)       AS bursary_flag_nulls,
        COUNT(*) - COUNT(payment_status)     AS payment_status_nulls,   
        COUNT(*) - COUNT(enrolment_status)   AS enrolment_status_nulls
    FROM finance
""").show()

┌───────────────────┬──────────────────┬─────────────────┬───────────┬──────────────────┬───────────────────┬────────────────────┬──────────────────────┬────────────────────────┐
│ student_ref_nulls │ first_name_nulls │ last_name_nulls │ dob_nulls │ fee_status_nulls │ tuition_fee_nulls │ bursary_flag_nulls │ payment_status_nulls │ enrolment_status_nulls │
│       int64       │      int64       │      int64      │   int64   │      int64       │       int64       │       int64        │        int64         │         int64          │
├───────────────────┼──────────────────┼─────────────────┼───────────┼──────────────────┼───────────────────┼────────────────────┼──────────────────────┼────────────────────────┤
│                 0 │                0 │               0 │         0 │                0 │                 0 │                  0 │                    0 │                      0 │
└───────────────────┴──────────────────┴─────────────────┴───────────┴──────────────────┴────────────────

In [6]:
# LMS
con.sql("""
    SELECT
        COUNT(*) - COUNT(lms_id)          AS lms_id_nulls,
        COUNT(*) - COUNT(username)        AS username_nulls,
        COUNT(*) - COUNT(programme)       AS programme_nulls,
        COUNT(*) - COUNT(modules_enrolled) AS modules_enrolled_nulls,
        COUNT(*) - COUNT(last_login_date) AS last_login_nulls,
        COUNT(*) - COUNT(account_status)  AS account_status_nulls
    FROM lms
""").show()

┌──────────────┬────────────────┬─────────────────┬────────────────────────┬──────────────────┬──────────────────────┐
│ lms_id_nulls │ username_nulls │ programme_nulls │ modules_enrolled_nulls │ last_login_nulls │ account_status_nulls │
│    int64     │     int64      │      int64      │         int64          │      int64       │        int64         │
├──────────────┼────────────────┼─────────────────┼────────────────────────┼──────────────────┼──────────────────────┤
│            0 │              0 │               0 │                      0 │             1347 │                  564 │
└──────────────┴────────────────┴─────────────────┴────────────────────────┴──────────────────┴──────────────────────┘



### Insights

- **SRS:** 1,511 null nationality values (~10%) — HESA compliance risk.
    Nationality is a mandatory HESA return field with no fallback in other systems.
- **Finance:** No nulls — clean across all fields.
- **LMS:** 1,347 null last_login_date values; 564 null account_status values.
    Context needed — to be investigated in value distributions.

### 3.2 Duplicate Checks

#### 3.2.1 Duplicate IDs

In [7]:
# SRS
con.sql("""
    SELECT 
        student_id,
        COUNT(*) AS student_id_frequency_count
    FROM srs
    GROUP BY student_id
    HAVING student_id_frequency_count > 1
""").show()

┌────────────┬────────────────────────────┐
│ student_id │ student_id_frequency_count │
│  varchar   │           int64            │
└────────────┴────────────────────────────┘
                  0 rows                 



In [8]:
# Finance
con.sql("""
    SELECT 
        student_ref,
        COUNT(*) AS student_ref_frequency_count
    FROM finance
    GROUP BY student_ref
    HAVING student_ref_frequency_count > 1
""").show()

┌─────────────┬─────────────────────────────┐
│ student_ref │ student_ref_frequency_count │
│   varchar   │            int64            │
└─────────────┴─────────────────────────────┘
                   0 rows                  



In [9]:
# LMS
con.sql("""
    SELECT 
        lms_id,
        COUNT(*) AS lms_id_frequency_count
    FROM lms
    GROUP BY lms_id
    HAVING lms_id_frequency_count > 1
""").show()

┌─────────┬────────────────────────┐
│ lms_id  │ lms_id_frequency_count │
│ varchar │         int64          │
└─────────┴────────────────────────┘
               0 rows             



#### 3.2.2 Duplicate Persons

In [10]:
# SRS
con.sql("""
    SELECT 
        first_name,
        last_name,
        date_of_birth,
        COUNT(*) AS occurrences
    FROM srs
    GROUP BY first_name, last_name, date_of_birth
    HAVING occurrences > 1
    ORDER BY occurrences DESC
""").show()

┌────────────┬────────────┬───────────────┬─────────────┐
│ first_name │ last_name  │ date_of_birth │ occurrences │
│  varchar   │  varchar   │     date      │    int64    │
├────────────┼────────────┼───────────────┼─────────────┤
│ Thomas     │ Price      │ 2002-12-23    │           2 │
│ Mason      │ Begum      │ 1998-10-29    │           2 │
│ Rory       │ Mohamed    │ 2005-06-24    │           2 │
│ Hannah     │ Richardson │ 1998-05-25    │           2 │
│ Mason      │ Lindqvist  │ 1999-06-25    │           2 │
│ Cerys      │ Evans      │ 2007-06-29    │           2 │
│ Pippa      │ Yusuf      │ 1999-07-24    │           2 │
│ Thomas     │ Owusu      │ 2005-10-12    │           2 │
│ Mason      │ Larsen     │ 2003-11-21    │           2 │
│ Megan      │ Hall       │ 1992-07-10    │           2 │
│   ·        │  ·         │     ·         │           · │
│   ·        │  ·         │     ·         │           · │
│   ·        │  ·         │     ·         │           · │
│ Cameron    │

In [11]:
# Finance
con.sql("""
    SELECT 
        first_name,
        last_name,
        date_of_birth,
        COUNT(*) AS occurrences
    FROM finance
    GROUP BY first_name, last_name, date_of_birth
    HAVING occurrences > 1
    ORDER BY occurrences DESC
""").show()

┌────────────┬───────────┬───────────────┬─────────────┐
│ first_name │ last_name │ date_of_birth │ occurrences │
│  varchar   │  varchar  │     date      │    int64    │
├────────────┼───────────┼───────────────┼─────────────┤
│ Harriet    │ Svensson  │ 1997-01-16    │           2 │
└────────────┴───────────┴───────────────┴─────────────┘



In [12]:
# LMS
con.sql("""
    SELECT 
        username,
        COUNT(*) AS occurrences
    FROM lms
    GROUP BY username
    HAVING occurrences > 1
    ORDER BY occurrences DESC
""").show()

┌──────────┬─────────────┐
│ username │ occurrences │
│ varchar  │    int64    │
└──────────┴─────────────┘
          0 rows        



### Insights

**3.2.1 — Duplicate IDs:**
  - No duplicate IDs found across any of the three systems. Each student identifier is unique within its own system.



**3.2.2 — Duplicate Persons:**
  - **SRS:** 76 name + DOB combinations appear more than once. Root cause unknown at this stage — requires investigation to determine whether these are migration artefacts, data entry errors, or coincidental matches.
  - **Finance:** 1 name + DOB match found — Requires investigation.
  - **LMS:** No duplicate usernames found.

### 3.3 Value Distributions
- 3.3.1 Enrolment / Account Status (SRS + Finance + LMS)  
- 3.3.2 Fee Status (SRS + Finance)
- 3.3.3 Programme (SRS + LMS)
- 3.3.4 Nationality (SRS) — null investigation
- 3.3.5 Last Login & Account Status (LMS) — null investigation

#### 3.3.1 Enrolment / Account Status (SRS + Finance + LMS)
- SRS, Finance and LMS all track student status but use different field names. Are they describing the same thing?
- SRS and Finance use enrolement_status, LMS uses account_status.

In [13]:
con.sql("""
    SELECT 
        'srs' AS system, 
        enrolment_status AS status, 
        COUNT(*) AS n       
    FROM srs     
    GROUP BY enrolment_status
    
    UNION ALL
    
    SELECT 
        'finance' AS system, 
        enrolment_status AS status, 
        COUNT(*) AS n       
    FROM finance     
    GROUP BY enrolment_status
    
    UNION ALL
    
    SELECT 
        'lms' AS system, 
        account_status AS status, 
        COUNT(*) AS n       
    FROM lms    
    GROUP BY account_status
""").show()

┌─────────┬───────────┬───────┐
│ system  │  status   │   n   │
│ varchar │  varchar  │ int64 │
├─────────┼───────────┼───────┤
│ srs     │ Withdrawn │  1513 │
│ srs     │ Active    │ 12348 │
│ srs     │ Suspended │   624 │
│ srs     │ Deferred  │   590 │
│ finance │ Inactive  │  1958 │
│ finance │ Enrolled  │ 12283 │
│ finance │ Deferred  │   587 │
│ finance │ Suspended │   622 │
│ lms     │ Inactive  │  1287 │
│ lms     │ Active    │ 12249 │
│ lms     │ NULL      │   564 │
└─────────┴───────────┴───────┘
  11 rows           3 columns



#### 3.3.2 — Fee Status (SRS + Finance)

- SRS and Finance both hold fee status. Are they using the same values to describe it?

In [14]:
con.sql("""
    SELECT 
        'srs' AS system,
        fee_status AS status,
        COUNT(*) AS occurences
    FROM srs
    GROUP BY status
    
    UNION ALL
    
    SELECT 
        'finance' AS system,
        fee_status AS status,
        COUNT(*) AS occurences
    FROM finance
    GROUP BY status

""").show()

┌─────────┬──────────┬────────────┐
│ system  │  status  │ occurences │
│ varchar │ varchar  │   int64    │
├─────────┼──────────┼────────────┤
│ srs     │ Home     │      10551 │
│ srs     │ Overseas │       4524 │
│ finance │ H        │      10603 │
│ finance │ O        │       4847 │
└─────────┴──────────┴────────────┘



#### 3.3.3 Programme (SRS + LMS)
- SRS uses programme_name. LMS uses a single programme field. Are these describing the same thing?

In [15]:
con.sql("""
    SELECT 
        'srs' AS system,
        programme_name AS programme,
        COUNT(*) AS n
    FROM srs
    GROUP BY programme_name

    UNION ALL

    SELECT 
        'lms' AS system,
        programme AS programme,
        COUNT(*) AS n
    FROM lms
    GROUP BY programme

    ORDER BY system, n DESC
""").show()

┌─────────┬─────────────────────────────┬───────┐
│ system  │          programme          │   n   │
│ varchar │           varchar           │ int64 │
├─────────┼─────────────────────────────┼───────┤
│ lms     │ Psychology (BSc)            │  1435 │
│ lms     │ Computer Science BSc        │  1434 │
│ lms     │ Business Management BSc     │  1432 │
│ lms     │ MBA                         │  1426 │
│ lms     │ Mathematics BSc             │  1410 │
│ lms     │ History BA                  │  1409 │
│ lms     │ English Literature BA       │  1401 │
│ lms     │ Nursing BSc                 │  1400 │
│ lms     │ Civil Engineering MEng      │  1383 │
│ lms     │ Data Science MSc            │  1370 │
│ srs     │ BSc Psychology              │  1533 │
│ srs     │ BSc Computer Science        │  1526 │
│ srs     │ BSc Business Management     │  1525 │
│ srs     │ BA History                  │  1525 │
│ srs     │ BSc Nursing                 │  1515 │
│ srs     │ MBA Business Administration │  1514 │


#### 3.3.4 Nationality (SRS) —  Null Investigation
- 1511 null nationality values identified in 3.1 — checking whether the non-null values are clean and what proportion of the population is affected.

In [16]:
con.sql("""
    SELECT 
        nationality,
        COUNT(*) AS n
    FROM srs
    GROUP BY nationality
    ORDER BY n DESC
""").show()

┌─────────────┬───────┐
│ nationality │   n   │
│   varchar   │ int64 │
├─────────────┼───────┤
│ GBR         │  9497 │
│ NULL        │  1511 │
│ CHN         │  1082 │
│ IND         │   663 │
│ NGA         │   401 │
│ DEU         │   281 │
│ PAK         │   277 │
│ IRN         │   277 │
│ USA         │   276 │
│ ZWE         │   276 │
│ ITA         │   268 │
│ MYS         │   266 │
└─────────────┴───────┘
  12 rows   2 columns



#### 3.3.5 Last Login & Account Status (LMS) — Null Investigation
- 1,347 null last_login_date and 564 null account_status values identified in 3.1 — investigating whether the two are related.

In [17]:
con.sql("""
    SELECT 
        account_status,
        COUNT(*) AS total,
        COUNT(*) - COUNT(last_login_date) AS last_login_nulls
    FROM lms
    GROUP BY account_status
    ORDER BY account_status
""").show()

┌────────────────┬───────┬──────────────────┐
│ account_status │ total │ last_login_nulls │
│    varchar     │ int64 │      int64       │
├────────────────┼───────┼──────────────────┤
│ Active         │ 12249 │                0 │
│ Inactive       │  1287 │             1287 │
│ NULL           │   564 │               60 │
└────────────────┴───────┴──────────────────┘



#### Insights
**3.3.1 — Enrolment / Account Status:**
- SRS and Finance both track enrolment status but use different labels — SRS uses "Active"/"Withdrawn", Finance uses "Enrolled"/"Inactive". Whether these are equivalent needs confirmation from stakeholders.  
- LMS account_status appears to be a simpler binary field — whether it maps to enrolment status or is managed independently needs clarification.

**3.3.2 — Fee Status:**
- SRS and Finance both track fee status but use different coding — SRS uses "Home"/"Overseas", Finance uses "H"/"O". Same concept, different format. Can't compare them directly without mapping one to the other.

**3.3.3 — Programme:**
- Naming convention — same programmes, different format. SRS puts the qualification first ("BSc Computer Science"), LMS puts the subject first ("Computer Science BSc"). One even has brackets — "Psychology (BSc)". No way to join on programme name without transformation first.  
- Count differences — LMS is consistently lower across every programme. Taken alongside the row count difference from section 2 (SRS 15,075 vs LMS 14,100), this suggests students exist in SRS with no LMS record. The count difference isn't random, it's systematic.

**3.3.4 — Nationality:**
- 1,511 null values (~10% of SRS) — a significant completeness gap. Non-null values are clean ISO codes (GBR, CHN, IND etc.) with no malformed entries. The issue is purely one of completeness, not value quality.

**3.3.5 — Last Login Date:**

The 1,347 null login dates from section 3.1 are not random — they map entirely to account status:

- **Inactive (1,287):** Every inactive student has a null login date — 100% correlation. Whether this reflects students who were deactivated before ever logging in, or whether login history is being cleared on deactivation, cannot be determined from the data alone. The pattern warrants investigation.
- **Null account_status — 60 of 564:** A subset of the 564 students with no account_status also have no login date. These are incomplete records with no recorded activity.
- **Null account_status — 504 of 564:** The remaining 504 have a null account_status but *do* have a login date. They have used the system, but their status was never set. The cause is unknown and requires investigation.

**Arithmetic check:** 1,287 (inactive) + 60 (null status, no login) = 1,347 — reconciles exactly with section 3.1.

## 4. Cross-System Analysis — ID Linkage & Population Overlaps

### Sampling ID formats
- Each system uses a different identifier field (SRS: student_id, Finance: student_ref, LMS: lms_id).
- Sampling values to confirm the format before attempting to join.

In [18]:
# DuckDB doesn't like LIMIT inside a UNION ALL, so created subqueries and selected * from them.
con.sql("""
    SELECT * FROM (SELECT 'srs' AS system, student_id AS id FROM srs LIMIT 5)
    
    UNION ALL
    
    SELECT * FROM (SELECT 'finance' AS system, student_ref AS id FROM finance LIMIT 5)
    
    UNION ALL
    
    SELECT * FROM (SELECT 'lms' AS system, lms_id AS id FROM lms LIMIT 5)
""").show()

┌─────────┬──────────┐
│ system  │    id    │
│ varchar │ varchar  │
├─────────┼──────────┤
│ srs     │ SRS00001 │
│ srs     │ SRS00002 │
│ srs     │ SRS00003 │
│ srs     │ SRS00004 │
│ srs     │ SRS00005 │
│ finance │ FIN00001 │
│ finance │ FIN00002 │
│ finance │ FIN00003 │
│ finance │ FIN00004 │
│ finance │ FIN00005 │
│ lms     │ s00001   │
│ lms     │ s00002   │
│ lms     │ s00003   │
│ lms     │ s00004   │
│ lms     │ s00005   │
└─────────┴──────────┘
  15 rows  2 columns



#### Insights
- Format: prefix + number accross all systems. 
- Can we strip the prefix and join on the numeric values?

### Validating the Join Logic
- Stripping the prefix and joining on the numeric component assumes that SRS00001 and FIN00001 refer to the same student.
- Before building cross-system analysis on this join, we need to verify it - if the same record appears in both systems, shared fields (first_name, last_name, date_of_birth) should agree.
- Sampling 20 matched records from SRS and Finance (both hold name and DOB) to confirm.

In [19]:
con.sql("""
    SELECT
        srs.student_id,
        finance.student_ref,
        srs.first_name        AS srs_first,    
        finance.first_name    AS fin_first,
        srs.last_name         AS srs_last,     
        finance.last_name     AS fin_last,
        srs.date_of_birth     AS srs_dob,      
        finance.date_of_birth AS fin_dob,
        srs.first_name    = finance.first_name    AS first_match,
        srs.last_name     = finance.last_name     AS last_match,
        srs.date_of_birth = finance.date_of_birth AS dob_match
    FROM srs
    JOIN finance ON REGEXP_REPLACE(srs.student_id, '[^0-9]', '', 'g') = REGEXP_REPLACE(finance.student_ref, '[^0-9]', '', 'g')
    LIMIT 10
""").show()

┌────────────┬─────────────┬───────────┬───────────┬──────────┬──────────┬────────────┬────────────┬─────────────┬────────────┬───────────┐
│ student_id │ student_ref │ srs_first │ fin_first │ srs_last │ fin_last │  srs_dob   │  fin_dob   │ first_match │ last_match │ dob_match │
│  varchar   │   varchar   │  varchar  │  varchar  │ varchar  │ varchar  │    date    │    date    │   boolean   │  boolean   │  boolean  │
├────────────┼─────────────┼───────────┼───────────┼──────────┼──────────┼────────────┼────────────┼─────────────┼────────────┼───────────┤
│ SRS00001   │ FIN00001    │ Henry     │ Henry     │ Ali      │ Ali      │ 1990-12-10 │ 1990-12-10 │ true        │ true       │ true      │
│ SRS00002   │ FIN00002    │ Ava       │ Ava       │ Kaur     │ Kaur     │ 2006-05-18 │ 2006-05-18 │ true        │ true       │ true      │
│ SRS00003   │ FIN00003    │ Reuben    │ Reuben    │ Conti    │ Conti    │ 1991-06-05 │ 1991-06-05 │ true        │ true       │ true      │
│ SRS00004   │ FIN00

In [20]:
con.sql("""
    WITH join_check AS (
        SELECT
            srs.first_name    = finance.first_name    AS first_match,
            srs.last_name     = finance.last_name     AS last_match,
            srs.date_of_birth = finance.date_of_birth AS dob_match
        FROM srs
        JOIN finance ON REGEXP_REPLACE(srs.student_id, '[^0-9]', '', 'g') = REGEXP_REPLACE(finance.student_ref, '[^0-9]', '', 'g')
    )
    SELECT
        COUNT(*)                        AS total_ids_matched,
        SUM(first_match::INT)           AS first_name_agree,
        SUM(last_match::INT)            AS last_name_agree,
        SUM(dob_match::INT)             AS dob_agree
    FROM join_check
""").show()
# SUM(first_match::INT) - booleans can't be summed directly in DuckDB, so ::INT casts true → 1 and false → 0. Summing them gives a count of how many were true

┌───────────────────┬──────────────────┬─────────────────┬───────────┐
│ total_ids_matched │ first_name_agree │ last_name_agree │ dob_agree │
│       int64       │      int128      │     int128      │  int128   │
├───────────────────┼──────────────────┼─────────────────┼───────────┤
│             15000 │            15000 │           15000 │     15000 │
└───────────────────┴──────────────────┴─────────────────┴───────────┘



### SRS vs Finance — match rate & population gaps

In [21]:
''' REGEXP_REPLACE takes four arguments:
  1. srs.student_id — the value to work on, e.g. SRS00001  
  2. '[^0-9]' — the pattern to find. The square brackets mean "any character in this set", and ^ inside brackets means "NOT". So [^0-9] = "any character that is not a digit"
  3. '' — replace whatever is found with nothing (empty string)  
  4. 'g' — do it globally, i.e. replace every match not just the first one
  So on SRS00001: finds S, R, S — all not digits — replaces each with nothing — leaves 00001.
'''

con.sql("""
    SELECT
        COUNT(*) AS srs_fin_matches
    FROM srs
    JOIN finance ON REGEXP_REPLACE(srs.student_id, '[^0-9]', '', 'g') = REGEXP_REPLACE(finance.student_ref, '[^0-9]', '', 'g')
""").show()

┌─────────────────┐
│ srs_fin_matches │
│      int64      │
├─────────────────┤
│           15000 │
└─────────────────┘



#### Insights
- SRS has 15,075 rows.
- Finance has 15,450 rows.
- Only 15,000 matches between systems.
- Therefore: 
    - 75 students are in SRS but not Finance — students with an SRS record but no financial record.
    - 450 records are in Finance but not SRS — financial records with no corresponding SRS record.

### SRS vs LMS — match rate & population gaps

In [22]:
con.sql("""
    SELECT
        COUNT(*) AS srs_lms_matches
    FROM srs
    JOIN lms ON REGEXP_REPLACE(srs.student_id, '[^0-9]', '', 'g') = REGEXP_REPLACE(lms.lms_id, '[^0-9]', '', 'g')
""").show()

┌─────────────────┐
│ srs_lms_matches │
│      int64      │
├─────────────────┤
│           14100 │
└─────────────────┘



#### Insights
- SRS has 15,075 rows.
- LMS has 14,100 rows.
- Only 14,100 matches between systems.
- Therefore:
    - 975 students are in SRS but not LMS — students with an SRS record but no LMS account.
    - 0 records are in LMS but not SRS — every LMS student has a corresponding SRS record.

### Finance vs LMS — match rate & population gaps

In [23]:
con.sql("""
    SELECT
        COUNT(*) AS finance_lms_matches
    FROM finance
    JOIN lms ON REGEXP_REPLACE(finance.student_ref, '[^0-9]', '', 'g') = REGEXP_REPLACE(lms.lms_id, '[^0-9]', '', 'g')
""").show()

┌─────────────────────┐
│ finance_lms_matches │
│        int64        │
├─────────────────────┤
│               14100 │
└─────────────────────┘



#### Insights
- Finance has 15,450 rows.
- LMS has 14,100 rows.
- Only 14,100 matches between systems.
- Therefore:
    - 1,350 records are in Finance but not LMS.
    - 0 records are in LMS but not Finance — every LMS student has a corresponding Finance record.

## 5. Worked Example — Root Cause Investigation

- SRS has 15,075 rows.
- Finance has 15,450 rows.
- Only 15,000 matches between systems.

### 5.1
- Investigating the 450 records that are in Finance but not SRS — financial records with no corresponding SRS record.


In [24]:
# Create view of the 450 records in Finance but not in SRS.
# Use LEFT JOIN because we want everything in finance.
# Filter to Nulls as we want the records that are not in SRS.

con.execute("""      
    CREATE OR REPLACE VIEW finance_not_in_srs AS
    SELECT finance.*      
    FROM finance
    LEFT JOIN srs ON REGEXP_REPLACE(finance.student_ref, '[^0-9]', '', 'g') = REGEXP_REPLACE(srs.student_id, '[^0-9]', '', 'g')
    WHERE srs.student_id IS NULL
  """)

In [25]:
con.sql("SELECT * FROM finance_not_in_srs LIMIT 20")

┌─────────────┬────────────┬───────────┬───────────────┬────────────┬────────────────────┬──────────────┬────────────────┬──────────────────┬───────────────┐
│ student_ref │ first_name │ last_name │ date_of_birth │ fee_status │ tuition_fee_amount │ bursary_flag │ payment_status │ enrolment_status │ academic_year │
│   varchar   │  varchar   │  varchar  │     date      │  varchar   │       int64        │   varchar    │    varchar     │     varchar      │    varchar    │
├─────────────┼────────────┼───────────┼───────────────┼────────────┼────────────────────┼──────────────┼────────────────┼──────────────────┼───────────────┤
│ FIN15076    │ Leon       │ Andersson │ 2002-02-14    │ H          │              18000 │ Y            │ Paid           │ Inactive         │ 2025/26       │
│ FIN15077    │ Isabella   │ Toure     │ 1990-01-17    │ O          │              15000 │ Y            │ Paid           │ Inactive         │ 2025/26       │
│ FIN15078    │ Rosa       │ Martin    │ 2007-11-19 

It appears that enrolment_status is inactive for all of these records. Verify that next.

In [26]:
con.sql("""
        SELECT 
            enrolment_status,
            COUNT(*) AS n
        FROM finance_not_in_srs 
        GROUP BY enrolment_status
""").show()

┌──────────────────┬───────┐
│ enrolment_status │   n   │
│     varchar      │ int64 │
├──────────────────┼───────┤
│ Inactive         │   450 │
└──────────────────┴───────┘



#### Insights
- All 450 Finance records with no SRS match have an enrolment status of Inactive.
- The root cause is unknown — two possibilities
    1. They were in SRS, got removed, but Finance wasn't cleaned up.
    2. They were added to Finance directly and never made it into SRS.
- Requires investigation to determine root cause.

### 5.2
- Investigating the 75 records that are in SRS but not Finance — SRS records with no corresponding Financial record.

In [27]:
# Create view of the 75 records in SRS but not Finance.
# LEFT JOIN keeps all SRS rows. 
# Filter to nulls on Finance side = no match found.

con.execute("""
    CREATE OR REPLACE VIEW srs_not_in_finance AS
    SELECT srs.*
    FROM srs
    LEFT JOIN finance ON REGEXP_REPLACE(srs.student_id, '[^0-9]', '', 'g') = REGEXP_REPLACE(finance.student_ref, '[^0-9]', '', 'g')
    WHERE finance.student_ref IS NULL
""")

In [28]:
con.sql("SELECT * FROM srs_not_in_finance LIMIT 20")

┌────────────┬────────────┬───────────┬───────────────┬─────────────┬──────────────────┬────────────────┬─────────────────────────────┬──────────────────┬────────────┬────────────────┬───────────────────┐
│ student_id │ first_name │ last_name │ date_of_birth │ nationality │     domicile     │ programme_code │       programme_name        │ enrolment_status │ fee_status │ enrolment_date │ expected_end_date │
│  varchar   │  varchar   │  varchar  │     date      │   varchar   │     varchar      │    varchar     │           varchar           │     varchar      │  varchar   │      date      │       date        │
├────────────┼────────────┼───────────┼───────────────┼─────────────┼──────────────────┼────────────────┼─────────────────────────────┼──────────────────┼────────────┼────────────────┼───────────────────┤
│ SRS15001   │ Pippa      │ Yusuf     │ 1999-07-24    │ GBR         │ England          │ MBA            │ MBA Business Administration │ Active           │ Home       │ 2023-09-22  

In [29]:
con.sql("""
    SELECT 
        enrolment_status, 
        COUNT(*) AS n
    FROM srs_not_in_finance
    GROUP BY enrolment_status
""").show()

┌──────────────────┬───────┐
│ enrolment_status │   n   │
│     varchar      │ int64 │
├──────────────────┼───────┤
│ Active           │    65 │
│ Suspended        │     2 │
│ Deferred         │     3 │
│ Withdrawn        │     5 │
└──────────────────┴───────┘



#### Insights

- Mixed enrolment statuses, unlike 5.1.
- The majority (65) are active — missing financial records for current students is a higher priority than leftover inactive records.
- All 75 require investigation but the active cohort is the most urgent.

#### TEST BIT - SRS NOT IN LMS

 975 students are in SRS but not LMS — students with an SRS record but no LMS account.

In [30]:
con.execute("""
    CREATE OR REPLACE VIEW srs_not_in_lms AS
    SELECT srs.*
    FROM srs
    LEFT JOIN lms ON REGEXP_REPLACE(srs.student_id, '[^0-9]', '', 'g') = REGEXP_REPLACE(lms.lms_id, '[^0-9]', '', 'g')
    WHERE lms.lms_id IS NULL
""")

In [31]:
con.sql("SELECT * FROM srs_not_in_lms LIMIT 10")

┌────────────┬────────────┬───────────┬───────────────┬─────────────┬──────────┬────────────────┬─────────────────────────┬──────────────────┬────────────┬────────────────┬───────────────────┐
│ student_id │ first_name │ last_name │ date_of_birth │ nationality │ domicile │ programme_code │     programme_name      │ enrolment_status │ fee_status │ enrolment_date │ expected_end_date │
│  varchar   │  varchar   │  varchar  │     date      │   varchar   │ varchar  │    varchar     │         varchar         │     varchar      │  varchar   │      date      │       date        │
├────────────┼────────────┼───────────┼───────────────┼─────────────┼──────────┼────────────────┼─────────────────────────┼──────────────────┼────────────┼────────────────┼───────────────────┤
│ SRS00006   │ Millie     │ Jackson   │ 1995-07-02    │ CHN         │ Overseas │ MSC_DATA       │ MSc Data Science        │ Deferred         │ Overseas   │ 2025-09-27     │ 2028-06-30        │
│ SRS00019   │ Maya       │ Jackson

In [32]:
con.sql("""
    SELECT 
        enrolment_status,
        COUNT(*)
    FROM srs_not_in_lms
    GROUP BY enrolment_status
""").show()

┌──────────────────┬──────────────┐
│ enrolment_status │ count_star() │
│     varchar      │    int64     │
├──────────────────┼──────────────┤
│ Withdrawn        │           32 │
│ Deferred         │          590 │
│ Active           │          335 │
│ Suspended        │           18 │
└──────────────────┴──────────────┘



## 6. Issue Register

| Issue | Notes | System(s) | Scale | Category | Priority |
|---|---|---|---|---|---|
| 75 SRS not in Finance | Mostly active students (65 of 75) — fees may not be collected. | SRS / Finance | 75 | Population gap | High |
| 450 Finance not in SRS | All inactive. Either removed from SRS without Finance being updated, or added to Finance and never made it into SRS. | Finance / SRS | 450 | Population gap | Medium |
| 975 SRS not in LMS | Students with SRS record but no LMS account. Root cause unknown. | SRS / LMS | 975 | Population gap | Medium |
| 1,350 Finance not in LMS | Students with Finance record but no LMS account. Root cause unknown. | Finance / LMS | 1,350 | Population gap | Medium |
| 1,511 missing nationality | HESA mandatory field with no fallback in other systems. Compliance risk. | SRS | 1,511 (~10%) | Completeness | High |
| 564 null account_status | 504 have login history but no status; 60 have neither. Note: 60 overlap with last login nulls. | LMS | 564 (~4%) | Completeness | Medium |
| 1,347 null last login dates | Maps to inactive accounts (1,287) and null account_status records (60). Root cause requires investigation. | LMS | 1,347 | Completeness | Low |
| Fee status coding mismatch | Same concept, different format. SRS: "Home"/"Overseas", Finance: "H"/"O". Cannot be compared without transformation. | SRS / Finance | All | Consistency | Medium |
| Programme naming mismatch | Same concept, different format. SRS: "BSc Computer Science", LMS: "Computer Science BSc". Cannot be compared without transformation. | SRS / LMS | All | Consistency | Medium |
| Enrolment status label mismatch | SRS: "Active"/"Withdrawn", Finance: "Enrolled"/"Inactive". Equivalence requires stakeholder confirmation. | SRS / Finance | All | Consistency | Medium |
| 76 duplicate persons | Same name + DOB appearing twice. Root cause unknown — migration artefact, data entry error, or coincidental match. | SRS | 76 | Uniqueness | Medium |